# Destination Wedding Planner Agent — Guided Demo
This notebook runs the GitHub-ready project one step at a time. Read `PROJECT_WALKTHROUGH.md` first.

## 1. Imports and build the agent
The external Kiwi MCP tools and the three specialist agents are constructed asynchronously.

In [ ]:
from langchain.messages import HumanMessage
from langgraph.types import Command
from wedding_planner import WeddingContext, build_wedding_agent, analyze_inspiration_image

agent = await build_wedding_agent()
context = WeddingContext()
config = {"configurable": {"thread_id": "notebook-demo-1"}}

## 2. Authenticate
Before authentication, wrap-style middleware exposes only the `authenticate` tool.

In [ ]:
login = await agent.ainvoke(
    {"messages": [HumanMessage(content="planner@example.com, wedding123")]},
    context=context,
    config=config,
)
print(login["messages"][-1].content)

## 3. Optional multimodal inspiration image
Uncomment and replace the path with your own PNG or JPEG.

In [ ]:
# style_summary = analyze_inspiration_image("my_wedding_inspiration.png")
# print(style_summary)

## 4. Ask the multi-agent system for a destination-wedding plan
The coordinator saves details and delegates to the Kiwi MCP flight agent, Tavily venue agent, and SQL music agent.

In [ ]:
plan = await agent.ainvoke(
    {"messages": [HumanMessage(content=(
        "Plan a romantic beach wedding in Cancun on October 10, 2027 "
        "for 80 guests with a $45,000 budget. Guests will travel from Dallas. "
        "Search round-trip flights for October 7 through October 12, 2027. "
        "The style is ivory, blush, tropical greenery, and candlelight. "
        "Find flight options, venue ideas, and romantic music ideas. "
        "Do not email yet."
    ))]},
    context=context,
    config=config,
)
print(plan["messages"][-1].content)

## 5. Draft an email and trigger human approval

In [ ]:
email_attempt = await agent.ainvoke(
    {"messages": [HumanMessage(content="Email the final destination-wedding plan to the client.")]},
    context=context,
    config=config,
)
email_attempt["__interrupt__"][0].value["action_requests"][0]["args"]

## 6. Approve the paused email tool call

In [ ]:
sent = await agent.ainvoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    context=context,
    config=config,
)
print(sent["messages"][-1].content)